In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

# 1. SETUP, GOOGLE DRIVE & KAGGLE API

In [ ]:
!pip install -q kagglehub

import os
import time
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

from sklearn.ensemble import IsolationForest, RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# Mount Google Drive to save our reports permanently
from google.colab import drive
drive.mount('/content/drive')

# Setup Drive Paths
PROJECT_DIR = '/content/drive/MyDrive/defense_eval_project'
REPORTS_DIR = os.path.join(PROJECT_DIR, 'reports')
os.makedirs(REPORTS_DIR, exist_ok=True)

# Visual styling for the PDF graphs
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (8, 6)

# Download the AI4I Predictive Maintenance Dataset directly from Kaggle
print("[INFO] Pulling dataset from Kaggle...")
dataset_dir = kagglehub.dataset_download("stephanmatzka/predictive-maintenance-dataset-ai4i-2020")
csv_path = os.path.join(dataset_dir, "ai4i2020.csv")

print(f"[SUCCESS] Environment ready. Data located at: {csv_path}")

# 2. DATA CLEANING

In [ ]:
def preprocess_ai4i_data(filepath):
    df = pd.read_csv(filepath)
    print(f"Original shape: {df.shape}")

    # 1. Drop identifiers (UDI and Product ID don't affect machine physics)
    df = df.drop(columns=['UDI', 'Product ID'])

    # 2. One-Hot Encode the 'Type' column (Low, Medium, High quality variants)
    # This turns 'Type' into binary columns so the math models can read it
    df = pd.get_dummies(df, columns=['Type'], drop_first=True)

    # 3. Create a Single Target Column for Diagnosis
    # The dataset has separate columns for Tool Wear Failure (TWF), Heat Dissipation (HDF), etc.
    conditions = [
        (df['TWF'] == 1),
        (df['HDF'] == 1),
        (df['PWF'] == 1),
        (df['OSF'] == 1),
        (df['RNF'] == 1)
    ]
    choices = ['Tool Wear', 'Heat Fail', 'Power Fail', 'Overstrain', 'Random Fail']

    # If none of the faults are 1, it defaults to 'Normal'
    df['Diagnosis'] = np.select(conditions, choices, default='Normal')


    # 4. Drop the original fault columns, otherwise the model will cheat.
    cols_to_drop = ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
    df = df.drop(columns=cols_to_drop, errors='ignore')

    return df

# Load and clean
clean_df = preprocess_ai4i_data(csv_path)
print(f"\n[INFO] Data Cleaned. Fault Distribution:\n{clean_df['Diagnosis'].value_counts()}")
display(clean_df.head())

# 3. TWO-STAGE PIPELINE TRAINING

In [ ]:
from imblearn.over_sampling import SMOTE

# Separate the sensors (X) from the answers (y)
X = clean_df.drop(columns=['Diagnosis'])
y = clean_df['Diagnosis']

# 80/20 Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- STAGE 1: ISOLATION FOREST ---
# We expect about 3.5% of this specific dataset to be anomalies
print("[INFO] Training Stage 1: Isolation Forest...")
iso_forest = IsolationForest(contamination=0.035, random_state=42)
iso_forest.fit(X_train)

# --- STAGE 2: DIAGNOSTIC MODELS (Random Forest) ---
print("[INFO] Training Stage 2 Diagnostic Models...")
# Create a mask to find ONLY the broken machines in the training set
fault_mask = y_train != 'Normal'
X_train_faults = X_train[fault_mask]
y_train_faults = y_train[fault_mask]


'''==========================================
             SMOTE Implementation
=========================================='''
# 1. Initialize SMOTE
# k_neighbors=3 means it looks at the 3 closest fault points to draw its lines
smote = SMOTE(random_state=42, k_neighbors=3)

# 2. Resample ONLY the training data that we filtered for faults
# This will generate synthetic examples so all fault types have equal representation
print(f"[INFO] Fault counts BEFORE SMOTE:\n{y_train_faults.value_counts()}")

X_train_faults_smote, y_train_faults_smote = smote.fit_resample(X_train_faults, y_train_faults)

print(f"[INFO] Fault counts AFTER SMOTE:\n{y_train_faults_smote.value_counts()}")



# Model A: Baseline Random Forest with Class Weighting
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
# Train the model on the new, synthetically balanced dataset
rf_model.fit(X_train_faults_smote, y_train_faults_smote)

# Model B: Lab Placeholder (Gradient Boosting)
lab_model = GradientBoostingClassifier(random_state=42)
lab_model.fit(X_train_faults, y_train_faults)

# Package them in a dictionary for easy evaluation later
diagnostic_models = {
    "Random Forest Baseline": rf_model,
    "Gradient Boosting (Lab Placeholder)": lab_model
}

print("[SUCCESS] All models trained and ready.")

# 4. EVALUATION & AUTOMATED PDF REPORT

In [ ]:
def run_evaluation_and_export_pdf(X_test, y_test, if_model, model_dict):
    pdf_path = os.path.join(REPORTS_DIR, 'Diagnostic_Model_Comparison_Report.pdf')

    # Create the PDF object
    with PdfPages(pdf_path) as pdf:

        # --- Run Stage 1 (Detect Anomalies) ---
        if_preds = if_model.predict(X_test)
        alarm_indices = (if_preds == -1)
        X_test_alarms = X_test[alarm_indices]

        # Loop through both models and compare
        for model_name, diagnostic_model in model_dict.items():
            final_preds = np.full(len(y_test), 'Normal', dtype=object)

            # Start Inference Timer
            start_time = time.time()

            if np.any(alarm_indices):
                # Stage 2 Prediction
                diagnoses = diagnostic_model.predict(X_test_alarms)
                final_preds[alarm_indices] = diagnoses

            # Stop Inference Timer
            end_time = time.time()
            inference_speed = (end_time - start_time) * 1000 # Convert to ms

            # Calculate F1 Score (Macro average handles class imbalance)
            f1 = f1_score(y_test, final_preds, average='macro', zero_division=0)
            report_str = classification_report(y_test, final_preds, zero_division=0)

            # 1. Create a Text Summary Page for the PDF
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.axis('off') # Hide the graph axes
            summary_text = (
                f"MODEL PERFORMANCE REPORT\n"
                f"========================\n\n"
                f"Algorithm: {model_name}\n"
                f"Total Inference Time: {inference_speed:.2f} ms\n"
                f"F1-Score (Macro): {f1:.4f}\n\n"
                f"Classification Report:\n{report_str}"
            )
            ax.text(0.01, 0.99, summary_text, fontsize=10, fontfamily='monospace',
                    verticalalignment='top', horizontalalignment='left')
            pdf.savefig(fig) # Save the text page to PDF
            plt.close()

            # 2. Create the Confusion Matrix Graph for the PDF
            labels = np.unique(np.concatenate((y_test, final_preds)))
            cm = confusion_matrix(y_test, final_preds, labels=labels)

            fig, ax = plt.subplots(figsize=(8, 6))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                        xticklabels=labels, yticklabels=labels, ax=ax)
            ax.set_title(f'Confusion Matrix: {model_name}')
            ax.set_ylabel('Actual Fault')
            ax.set_xlabel('Predicted Fault')
            plt.xticks(rotation=45)

            pdf.savefig(fig, bbox_inches='tight') # Save the graph to PDF
            plt.close()

            # Also print to Colab screen so you can see it immediately
            print(summary_text)
            print("-" * 50)

    print(f"\n[SUCCESS] Executive PDF Report generated at: {pdf_path}")

# Run the full pipeline
run_evaluation_and_export_pdf(X_test, y_test, iso_forest, diagnostic_models)

# 5. EXPLAINABLE AI: FEATURE IMPORTANCE

In [ ]:
def plot_feature_importance(model, feature_names):
    # Extract the mathematical weights the model assigned to each sensor
    importances = model.feature_importances_

    # Sort them from least to most important
    indices = np.argsort(importances)

    plt.figure(figsize=(10, 6))
    plt.title('Explainable AI: Sensor Importance for Fault Diagnosis', fontsize=14)

    # Create a horizontal bar chart
    plt.barh(range(len(indices)), importances[indices], color='steelblue', align='center')
    plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
    plt.xlabel('Relative Importance (Gini Importance)', fontsize=12)
    plt.ylabel('Sensor / Feature', fontsize=12)

    # Save the graph to your Drive so you can show Dr. Dick tomorrow
    fi_path = os.path.join(REPORTS_DIR, 'Feature_Importance_Report.png')
    plt.savefig(fi_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"[SUCCESS] Interpretability report saved to: {fi_path}")

# Run the function using your trained Random Forest and the sensor column names
plot_feature_importance(rf_model, X_train_faults.columns)